In [ ]:
import time
_T_NB0 = time.perf_counter()   # notebook timer (read again after the model run)

# Make the "opendrift" conda env behave as if it were activated.
# VS Code sometimes launches the kernel without activation, which leaves GDAL_DATA unset and
# lets the wrong HDF5/GDAL DLLs load (kernel dies with exit code 0xC06D007F). Harmless if already activated.
import os, sys
from pathlib import Path

if sys.platform == "win32":
    _env = Path(sys.executable).resolve().parent
    for _sub in ["Library/mingw-w64/bin", "Library/usr/bin", "Library/bin", "Scripts", ""]:
        _p = _env / _sub if _sub else _env
        if _p.is_dir():
            if str(_p) not in os.environ.get("PATH", ""):
                os.environ["PATH"] = str(_p) + os.pathsep + os.environ.get("PATH", "")
            try:
                os.add_dll_directory(str(_p))
            except (OSError, AttributeError):
                pass
    os.environ.setdefault("CONDA_PREFIX", str(_env))
    if (_env / "Library/share/gdal").is_dir():
        os.environ.setdefault("GDAL_DATA", str(_env / "Library/share/gdal"))
    if (_env / "Library/share/proj").is_dir():
        os.environ.setdefault("PROJ_DATA", str(_env / "Library/share/proj"))
        os.environ.setdefault("PROJ_LIB", str(_env / "Library/share/proj"))

print("python   :", sys.executable)
print("GDAL_DATA:", os.environ.get("GDAL_DATA"))

# OpenDrift coral larvae for GBR waters using operational 3D ROMS and GFS winds

In [ ]:
import opendrift
import importlib.metadata

print("OpenDrift:", importlib.metadata.version("opendrift"))
print("Location:", opendrift.__file__)

In [ ]:
import sys
from pathlib import Path

PROJECT_DIR = Path(r"D:\Thesis")
MODELS_DIR = PROJECT_DIR / "models"

sys.path.insert(0, str(MODELS_DIR))

In [ ]:
import os
import random
import sys
import datetime

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd

from datetime import timedelta
from netCDF4 import Dataset
from opendrift.readers import reader_ROMS_native
from opendrift.readers import reader_netCDF_CF_generic
from opendrift.readers import reader_shape
from opendrift.models.oceandrift import OceanDrift
from shapely.geometry import Polygon, Point

from AcroporaM_Final import LarvalCorals

In [ ]:
import pyproj
print(pyproj.datadir.get_data_dir())

In [ ]:
# Batch parameters
run_id = "S3-Def"
vertical_behavior_mode = "S3"

duration = 30
number_particles = 100

windage_factor = 0.03
time_step_minutes = 60          # model time step (must divide 60); output is always hourly
competency_start_days = 4.0     # "Pre-Comp": days before larvae become competent to settle
swim_speed = 0.00041            # m/s, active swimming speed used by the vertical behaviours


### Simulation Date and Time

In [ ]:
t0 = datetime.datetime(2025, 11, 10, 10, 0, 0)
ndays = duration + 1

model_day = []
for d in range(0,ndays):
        model_day.append(t0 + datetime.timedelta(days = d))

### Load ROMS 

In [ ]:
# Define location of the ROMS model
roms_files = []
for d in model_day:
    roms_files.append('/Thesis/data/roms/gbr_his_%s.nc' %(d.strftime('%Y%m%d')))

print("ROMS files expected:")
for f in roms_files:
    print(" ", f, "  exists:", os.path.isfile(f))

gfs_location = '/Thesis/data/gfs/GFS_AUS_20251xxx.nc'
gfs = reader_netCDF_CF_generic.Reader(gfs_location, standard_name_mapping={'Uwind': 'x_wind', 'Vwind': 'y_wind'})
roms = reader_ROMS_native.Reader(roms_files, standard_name_mapping={
        "h": "sea_floor_depth_below_sea_level"
    })

reader_landmask_local = reader_shape.Reader.from_shpfiles('/Thesis/data/spatial/GBR_polygons.shp')

In [ ]:
with xr.open_dataset(roms_files[0]) as test_roms:
    print(list(test_roms.data_vars))

### Configuration

In [ ]:
o = LarvalCorals(loglevel=20)  # loglevel=20 gives INFO, use 50 for WARNING only
o.add_reader([reader_landmask_local, roms, gfs])
#o.add_reader([roms, gfs])

# ── Physics ──
o.set_config('drift:advection_scheme', 'runge-kutta4')
o.set_config('drift:horizontal_diffusivity', 0.05)
o.set_config('vertical_mixing:background_diffusivity', 1e-5)
o.set_config('vertical_mixing:diffusivitymodel', 'windspeed_Large1994')
o.set_config('drift:vertical_mixing', True)
o.set_config('drift:vertical_mixing_at_surface', True)
o.set_config('windage_factor', windage_factor)
o.set_config('windage_depth', 0.5)

# ── Boundaries ──
o.set_config('general:coastline_action', 'previous')
o.set_config('general:use_auto_landmask', True)   # use our high-res local mask
o.set_config('seed:ocean_only', True)

# ── Uncertainty ──
o.set_config('drift:current_uncertainty', 0.01)
o.set_config('drift:wind_uncertainty', 0.5)

# ── Biology ──
o.set_config('biology:competency_start_days', competency_start_days)
o.set_config('biology:competency_end_days', 30)
o.set_config('biology:mortality_rate_per_day', 0.054)
o.set_config("biology:settlement_depth", 2.0)
o.set_config("biology:vertical_behavior_mode", vertical_behavior_mode)
o.set_config("swim_speed", swim_speed)

o.max_speed = 3  # cap unrealistically fast currents at 3 m/s


In [ ]:
def random_points_in_polygon(number, polygon):
    """Return `number` random (lon, lat) Point objects inside `polygon`."""
    points = []
    min_x, min_y, max_x, max_y = polygon.bounds
    i = 0
    while i < number:
        point = Point(random.uniform(min_x, max_x), random.uniform(min_y, max_y))
        if polygon.contains(point):
            points.append(point)
            i += 1
    return points

### Spawning

In [ ]:
gbr = gpd.read_file('/Thesis/data/spatial/SeamapAus_Clipped.shp')

reef_names = ["Moore", "Elford", "Thetford", "Milln"]

partial_match = gbr["GBR_NAME"].str.contains(
    "|".join(reef_names),
    case=False,
    na=False,
)

briggs_match = gbr["GBR_NAME"].str.strip().str.casefold().eq("briggs reef")

reefs = gbr[partial_match | briggs_match].copy()

print("Number of reefs selected:", len(reefs))
print(reefs[["GBR_NAME", "OBJECTID"]])

moore_points = gpd.read_file('/Thesis/data/spatial/MooreCluster.shp')

print("Moore Cluster CRS:", moore_points.crs)
print("Number of spawning points:", len(moore_points))

if moore_points.crs != reefs.crs:
    moore_points = moore_points.to_crs(reefs.crs)

moore_points["source_point_id"] = np.arange(
    1,
    len(moore_points) + 1
)

print("Reef CRS:", gbr.crs)
print("Reef bounds:", gbr.total_bounds)

spawn_points = gpd.sjoin(moore_points, reefs[["GBR_NAME", "OBJECTID", "geometry"]], how="inner", predicate="within")
spawn_points = spawn_points.rename(
    columns={
        "OBJECTID": "source_reef_id",
        "GBR_NAME": "source_reef_name"
    }
)

if "index_right" in spawn_points.columns:
    spawn_points = spawn_points.drop(
        columns="index_right"
    )

# Check how many points belong to each reef
print("\nSpawn point columns:")
print(spawn_points.columns)

print("\nPoints per reef:")
print(
    spawn_points.groupby(
        ["source_reef_name"]
    ).size()
)

print("\nFirst 10 spawning points:")
print(
    spawn_points[
        [
            "source_point_id",
            "source_reef_id",
            "source_reef_name"
        ]
    ].head(10)
)


In [ ]:
import numpy as np

print("ROMS lon min:", float(np.nanmin(roms.lon)))
print("ROMS lon max:", float(np.nanmax(roms.lon)))
print("ROMS lat min:", float(np.nanmin(roms.lat)))
print("ROMS lat max:", float(np.nanmax(roms.lat)))


In [ ]:
print(spawn_points.columns)

In [ ]:
spawning_start_utc = datetime.datetime(2025, 11, 10, 10, 0, 0)
spawning_end_utc = datetime.datetime(2025, 11, 10, 18, 0, 0)

release_times = [
    spawning_start_utc + timedelta(minutes=minute)
    for minute in range(0, 181, 3)]

total_particles = number_particles
n_sources = len(reefs)

assert n_sources > 0, "No source reefs found"
assert total_particles % n_sources == 0, (
    "Choose a total particle number divisible by the number of source reefs.")

particles_per_reef = total_particles // n_sources  # 125 with eight reefs

print(f"\nTotal particles: {total_particles}")
print(f"Particles per reef: {particles_per_reef}")
print(f"Number of reefs: {n_sources}")

for _, reef in reefs.iterrows():

    source_id = int(reef["OBJECTID"])
    source_name = reef["GBR_NAME"]

    reef_points = spawn_points[spawn_points["source_reef_id"] == source_id].copy()

    assert len(reef_points) > 0, (
        f"No spawning points found for "
        f"{source_name} ({source_id})"
    )

    release_batches = np.array_split(
        np.arange(particles_per_reef),
        len(release_times),)

    for release_time, batch in zip(release_times, release_batches):
        n_this_release = len(batch)

        if n_this_release == 0:
            continue

        chosen_points = reef_points.sample(
            n=n_this_release,
            replace=True
        )

        #points = random_points_in_polygon(n_this_release, reef.geometry)

        o.seed_elements(
            lon=chosen_points.geometry.x.tolist(),
            lat=chosen_points.geometry.y.tolist(),
            z=-1,
            time=release_time,
            terminal_velocity=0.0,
            source_reef_id=source_id,
            source_point_id=chosen_points[
                "source_point_id"
            ].astype(int).tolist(),
        ) 

print("\nSpawning complete.")
print(reefs[["GBR_NAME", "OBJECTID"]])
print(f"Particles per reef: {particles_per_reef}")


In [ ]:
print("Total elements:", o.num_elements_total)
print("Active elements:", o.num_elements_active)
print("Scheduled elements:", o.num_elements_scheduled)
print("Deactivated elements:", o.num_elements_deactivated)

print("\nIDs:")
print(o.elements.ID[:20])

In [ ]:
o.set_reef_polygons(
    gbr,
    reef_id_column='OBJECTID',
    reef_name_column='GBR_NAME'
)

print(o.env.priority_list.get("sea_floor_depth_below_sea_level"))

In [ ]:
from datetime import timedelta

print("="*70)
print("ROMS TIME CHECK")
print("="*70)

print("ROMS start:", roms.start_time)
print("ROMS end:  ", roms.end_time)

print("\nSimulation start:", o.start_time)

simulation_duration = timedelta(days=ndays-1)
simulation_end = o.start_time + simulation_duration

print("Simulation duration:", simulation_duration)
print("Expected simulation end:", simulation_end)

print("\nROMS covers simulation?")

if roms.start_time <= o.start_time and roms.end_time >= simulation_end:
    print("YES - ROMS covers the entire simulation")
else:
    print("NO - ROMS does NOT cover the entire simulation")

    if roms.start_time > o.start_time:
        print("PROBLEM: ROMS starts AFTER the simulation starts")

    if roms.end_time < simulation_end:
        print("PROBLEM: ROMS ends BEFORE the simulation finishes")

        print("\nROMS ends:", roms.end_time)
        print("Simulation needs until:", simulation_end)

        print(
            "Missing time:",
            simulation_end - roms.end_time
        )

### Run Model

In [ ]:
import json, time

os.makedirs("results", exist_ok=True)
tag = f"{run_id}"
outfile = f"results/ensemble_{tag}.nc"

time_step_s = int(round(time_step_minutes * 60))
assert 3600 % time_step_s == 0, "time_step_minutes must divide 60 so every hourly output lands on a model step"

_t_run0 = time.perf_counter()
o.run(duration=datetime.timedelta(days=ndays-1),
      time_step = time_step_s,
      time_step_output = 3600,
      export_variables=['status',
                        'age_seconds',
                        'z',
                        'competent',
                        'settled',
                        'source_reef_id',
                        'settlement_reef_id',
                        'source_point_id',
                        'ocean_vertical_diffusivity',
                        'upward_sea_water_velocity',
                        'sea_floor_depth_below_sea_level',
                        'terminal_velocity'],
      outfile = outfile)
_t_run1 = time.perf_counter()

In [ ]:
# Run record: parameters + timings, read by Analysis_Postprocessing
runinfo = dict(
    run_id=run_id, scenario=vertical_behavior_mode,
    number_particles=number_particles, time_step_minutes=time_step_minutes,
    competency_start_days=competency_start_days, windage_factor=windage_factor,
    swim_speed=swim_speed, duration_days=duration,
    model_run_s=round(_t_run1 - _t_run0, 1),                                   # o.run() only
    notebook_s=round(_t_run1 - globals().get("_T_NB0", _t_run0), 1),           # ROMS load + seeding + o.run()
    finished_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
)
with open(f"results/runinfo_{tag}.json", "w") as f:
    json.dump(runinfo, f, indent=2)
print(f"{tag}: o.run() took {runinfo['model_run_s']/60:.1f} min (notebook total {runinfo['notebook_s']/60:.1f} min)")

In [ ]:
# Post-processing and GIS exports
# Outputs are named by vertical-behaviour scenario and written to results/.
import os
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

os.makedirs("results", exist_ok=True)
scenario = str(o.get_config("biology:vertical_behavior_mode"))
output_prefix = os.path.join("results", f"ensemble_{run_id}")
ds = o.result
# Preserve the original GBR map extent: west, east, south, north.
bbox = (143.5, 150.5, -21, -14)

# Particle-level metadata: use the first valid source ID and first settlement event.
def first_valid_id(values):
    values = np.asarray(values)
    valid = values[np.isfinite(values) & (values >= 0)]
    return int(valid[0]) if len(valid) else None

n_trajectories = ds.sizes["trajectory"]
source_id = np.array([
    first_valid_id(ds["source_reef_id"].isel(trajectory=i).values)
    for i in range(n_trajectories)
], dtype=object)

settled = ds["settled"].values == 1
first_settle_index = np.full(n_trajectories, -1, dtype=int)
has_settled = settled.any(axis=1)
first_settle_index[has_settled] = settled[has_settled].argmax(axis=1)

settlement_id = np.full(n_trajectories, -1, dtype=int)
for i in np.where(has_settled)[0]:
    ids = ds["settlement_reef_id"].isel(trajectory=i).values
    event_id = ids[first_settle_index[i]]
    if np.isfinite(event_id) and event_id >= 0:
        settlement_id[i] = int(event_id)
    else:
        valid_ids = ids[np.isfinite(ids) & (ids >= 0)]
        if len(valid_ids):
            settlement_id[i] = int(valid_ids[0])

particle_table = pd.DataFrame({
    "source_id": source_id,
    "settlement_id": settlement_id,
    "settled": has_settled,
})
particle_table = particle_table.dropna(subset=["source_id"])
particle_table["source_id"] = particle_table["source_id"].astype(int)

reef_info = gbr[["OBJECTID", "GBR_NAME", "geometry"]].drop_duplicates("OBJECTID").copy()
reef_info["OBJECTID"] = reef_info["OBJECTID"].astype(int)
reef_names = reef_info.set_index("OBJECTID")["GBR_NAME"]

In [ ]:
# ---------------------------------------------------------------------------
# A. Depth versus time
# ---------------------------------------------------------------------------
times = pd.to_datetime(ds.time.values)
z = ds["z"].values
fig, ax = plt.subplots(figsize=(13, 7))
for i in range(n_trajectories):
    valid = np.isfinite(z[i])
    if valid.any():
        ax.plot(times[valid], z[i, valid], alpha=0.10, linewidth=0.45)
settled_depths = z[np.arange(n_trajectories)[has_settled], first_settle_index[has_settled]]
ax.scatter(times[first_settle_index[has_settled]], settled_depths,
           marker="x", s=25, color="crimson", label="First settlement", zorder=3)
ax.set(xlabel="Time (UTC)", ylabel="Depth (m; negative below sea level)",
       title=f"Larval depth through time — {scenario}")
ax.grid(alpha=0.25)
if has_settled.any(): ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(f"{output_prefix}_depth_vs_time.png", dpi=300, bbox_inches="tight")
plt.show()